# Notebook 04 — Cross-Sectional Analysis: Traditional vs LLM-Computed CARs

This is the main analysis notebook where the hypotheses are tested. Three things are done:

1. **Descriptive statistics**: mean, median, std of CARs by event type and method (traditional vs LLM)
2. **G-Rank significance tests**: testing whether the mean/median CAR is significantly different from zero. The generalized rank test (Kolari & Pynnönen, 2011) is used, which is more robust than the simple t-test for event studies because it accounts for cross-sectional correlation and event-induced variance.
3. **Paired comparison tests**: testing whether there's a significant difference between the traditional and LLM-computed CARs (paired t-test + Wilcoxon signed-rank test).

The G-Rank test works by standardizing each event's CAR using its own estimation-window variance (giving SCARs), then ranking them against the estimation-period SARs. This is pretty involved but it's the state-of-the-art for event study significance testing.

**Inputs:**
- `cumulative_abnormal_returns_{mode}.csv`: CARs from notebook 03
- `portfolio_returns.csv`, `market_returns.csv`: for SAR computation in the G-Rank test

**Outputs:**
- `cross_sectional_results.csv`: all test results in one file (descriptive stats, G-Rank tests, comparison tests)

In [1]:
from collections import Counter
import numpy as np
import pandas as pd
from scipy import stats


CAR_COLUMNS = [
    "CAR_pre20_pre1",
    "CAR_0_1",
    "CAR_0_5",
    "CAR_0_10",
    "CAR_0_20",
    "CAR_pre20_20",
]

CAR_PERIOD_MAP = {
    "CAR_pre20_pre1": (-20, -1),
    "CAR_0_1": (0, 1),
    "CAR_0_5": (0, 5),
    "CAR_0_10": (0, 10),
    "CAR_0_20": (0, 20),
    "CAR_pre20_20": (-20, 20),
}

# L2 = number of trading days in each event window (inclusive of both endpoints)
CAR_L2_MAP = {
    "CAR_pre20_pre1": 20,
    "CAR_0_1": 2,
    "CAR_0_5": 6,
    "CAR_0_10": 11,
    "CAR_0_20": 21,
    "CAR_pre20_20": 41,
}

GROUP_DEFINITIONS = {
    "5a": ("P - Purchase", "traditional"),
    "5b": ("S - Sale", "traditional"),
    "5b'": ("S - Sale+OE", "traditional"),
    "5c": ("P - Purchase", "llm"),
    "5d": ("S - Sale", "llm"),
    "5d'": ("S - Sale+OE", "llm"),
}

MERGED_GROUP_DEFINITIONS = {
    "5a": ("P - Purchase", "traditional"),
    "5b": ("S - Sale+All", "traditional"),
    "5c": ("P - Purchase", "llm"),
    "5d": ("S - Sale+All", "llm"),
}

COMPARISON_PAIRS = [
    ("5a", "5c", "P - Purchase"),
    ("5b", "5d", "S - Sale"),
    ("5b'", "5d'", "S - Sale+OE"),
]

MERGED_COMPARISON_PAIRS = [
    ("5a", "5c", "P - Purchase"),
    ("5b", "5d", "S - Sale+All"),
]

## Setup and Constants

The CAR windows, group definitions (which event type + method combos to test), and the comparison pairs for the traditional-vs-LLM tests are defined here. The `L2` map stores the number of trading days in each window, which is needed for the SAR standardization in the G-Rank test.

In [2]:
DUPLICATE_MODE_MAP = {
    "both": (GROUP_DEFINITIONS, COMPARISON_PAIRS),
    "exclude": (GROUP_DEFINITIONS, COMPARISON_PAIRS),
    "merge": (MERGED_GROUP_DEFINITIONS, MERGED_COMPARISON_PAIRS),
}


def load_data(duplicate_handling="both"):
    car_path = f"cumulative_abnormal_returns_{duplicate_handling}.csv"
    combined = pd.read_csv(car_path, parse_dates=["event_date"])
    print(f"Loaded {len(combined)} rows from {car_path}")

    trad_df = combined[combined["source"] == "traditional"].copy()
    llm_df = combined[combined["source"] == "llm"].copy()
    print(f"  Traditional: {len(trad_df)} rows, LLM: {len(llm_df)} rows")

    # portfolio and market returns (for SAR calculation)
    portfolio_df = pd.read_csv("portfolio_returns.csv", parse_dates=["date"])
    market_df = pd.read_csv("market_returns.csv", parse_dates=["date"])

    group_defs, comp_pairs = DUPLICATE_MODE_MAP[duplicate_handling]

    return trad_df, llm_df, portfolio_df, market_df, group_defs, comp_pairs

## Data Loading

The pre-computed CARs and raw return data are loaded. The duplicate handling mode controls which version of the CARs is used (both/exclude/merge).

In [3]:
def descriptive_stats(df, car_columns, label):
    rows = []
    for col in car_columns:
        if col not in df.columns:
            continue
        series = df[col].dropna()
        rows.append({
            "group": label,
            "car_window": col,
            "N": len(series),
            "mean": series.mean(),
            "median": series.median(),
            "std": series.std(),
            "min": series.min(),
            "max": series.max(),
        })
    return rows

In [4]:
# G-Rank Significance Tests

def _build_trading_calendar(market_df):
    return np.sort(market_df["date"].dropna().unique())


def _trading_day_offset(trading_cal, event_date, offset):
    idx = np.searchsorted(trading_cal, np.datetime64(event_date), side="right") - 1
    if idx < 0:
        idx = 0
    target = idx + offset
    target = max(0, min(target, len(trading_cal) - 1))
    return trading_cal[target]


def calculate_SAR(ticker, event_date, portfolio_df, market_df, alpha, beta,
                  trading_cal, estimation_window=(-220, -21)):
    est_start = _trading_day_offset(trading_cal, event_date, estimation_window[0])
    est_end = _trading_day_offset(trading_cal, event_date, estimation_window[1])

    est_data = portfolio_df[
        (portfolio_df["date"] >= est_start) & (portfolio_df["date"] <= est_end)
    ].copy()

    if ticker not in est_data.columns:
        return None, None

    market_data = market_df[
        (market_df["date"] >= est_start) & (market_df["date"] <= est_end)
    ].copy()

    merged = pd.merge(
        est_data[["date", ticker]], market_data[["date", "market_return"]], on="date"
    )

    merged["AR"] = merged[ticker] - (alpha + beta * merged["market_return"])
    s_i = merged["AR"].std()  # ddof=1 by default

    if s_i == 0 or np.isnan(s_i):
        return None, None

    merged["SAR"] = merged["AR"] / s_i
    return merged[["date", "SAR"]], s_i


def rank_test(results_df, portfolio_df, market_df, car_col, alpha_col, beta_col,
              trading_cal):
    valid = results_df.dropna(subset=[car_col]).copy()
    N = len(valid)
    if N == 0:
        return []

    use_z_test = N < 30
    test_name = "G-Rank Z test" if use_z_test else "G-Rank t-test"

    L2 = CAR_L2_MAP[car_col]
    CAR_values = valid[car_col].values

    # collect SAR data and s_i per event
    L1_candidates = []
    valid_list = []

    for idx, (_, row) in enumerate(valid.iterrows()):
        ticker = row["ticker"]
        event_date = pd.to_datetime(row["event_date"])
        alpha = row[alpha_col]
        beta = row[beta_col]

        SAR_data, s_i = calculate_SAR(
            ticker, event_date, portfolio_df, market_df, alpha, beta, trading_cal
        )

        if SAR_data is not None and len(SAR_data) >= 30:
            L1_candidates.append(len(SAR_data))
            valid_list.append((idx, row, SAR_data, s_i))

    if not L1_candidates:
        return []

    L1 = Counter(L1_candidates).most_common(1)[0][0]

    # compute proper SCAR_i = CAR_i / (s_i * sqrt(L2)) for each event
    SCAR_values = np.full(len(CAR_values), np.nan)
    for idx, row, SAR_data, s_i in valid_list:
        SCAR_values[idx] = CAR_values[idx] / (s_i * np.sqrt(L2))

    # cross-sectional standardization per Kolari & Pynnönen [11.2]
    SCAR_mean = np.nanmean(SCAR_values)
    S_SCAR_sq = np.nansum((SCAR_values - SCAR_mean)**2) / (np.sum(~np.isnan(SCAR_values)) - 1)
    S_SCAR = np.sqrt(S_SCAR_sq)

    if S_SCAR == 0 or np.isnan(S_SCAR):
        return []

    results = []
    for test_type in ["mean", "median"]:
        if test_type == "mean":
            # SCAR*_i = SCAR_i / S_SCAR  (standardize, no centering)
            SCAR_event = SCAR_values / S_SCAR
        else:
            # for median test: center at median, then standardize
            SCAR_median = np.nanmedian(SCAR_values)
            SCAR_event = (SCAR_values - SCAR_median) / S_SCAR

        U_matrix = []
        valid_indices = []

        for idx, row, SAR_data, s_i in valid_list:
            SAR_vals = SAR_data["SAR"].values
            if len(SAR_vals) != L1:
                continue
            if np.isnan(SCAR_event[idx]):
                continue

            # GSAR = [SAR_1, ..., SAR_L1, SCAR_i]
            GSAR = np.append(SAR_vals, SCAR_event[idx])
            ranks = stats.rankdata(GSAR)
            U_i = ranks / (L1 + 2) - 0.5
            U_matrix.append(U_i)
            valid_indices.append(idx)

        if not U_matrix:
            continue

        U_matrix = np.array(U_matrix)
        N_valid = U_matrix.shape[0]
        U_bar = np.mean(U_matrix, axis=0)
        U_bar_event = U_bar[-1]

        if use_z_test:
            S_sq = L1 / (12 * N_valid * (L1 + 2))
            S = np.sqrt(S_sq)
            if S == 0:
                continue
            test_stat = U_bar_event / S
            p_value = 2 * (1 - stats.norm.cdf(abs(test_stat)))
            stat_name = "z_statistic"
            df = None
        else:
            S_sq = np.mean(U_bar ** 2)
            S = np.sqrt(S_sq)
            if S == 0:
                continue
            Z = U_bar_event / S
            denom = L1 - Z ** 2
            if denom <= 0:
                denom = abs(denom)
            test_stat = Z * np.sqrt((L1 - 1) / denom)
            df = L1 - 1
            p_value = 2 * (1 - stats.t.cdf(abs(test_stat), df))
            stat_name = "t_statistic"

        valid_CAR = CAR_values[valid_indices]
        if test_type == "mean":
            CAR_center = np.mean(valid_CAR)
        else:
            CAR_center = np.median(valid_CAR)

        results.append({
            "test_type": test_type,
            "test_method": test_name,
            "car_window": car_col,
            "N_total": N,
            "N_valid": N_valid,
            "L1": L1,
            "CAR_center": CAR_center,
            "CAR_std": np.std(valid_CAR, ddof=1),
            stat_name: test_stat,
            "degrees_of_freedom": df if df is not None else "N/A",
            "p_value": p_value,
            "significant_5pct": "Yes" if p_value < 0.05 else "No",
            "significant_1pct": "Yes" if p_value < 0.01 else "No",
        })

    return results


def run_grank_tests(df, portfolio_df, market_df, alpha_col, beta_col, group_label,
                    trading_cal):
    all_results = []
    for car_col in CAR_COLUMNS:
        if car_col not in df.columns:
            continue
        test_results = rank_test(
            df, portfolio_df, market_df, car_col, alpha_col, beta_col, trading_cal
        )
        for r in test_results:
            r["group"] = group_label
        all_results.extend(test_results)
    return all_results

## G-Rank Significance Tests

This is the core statistical test. The G-Rank test (Kolari & Pynnönen, 2011) is a non-parametric rank-based test specifically designed for event studies. It handles cross-sectional correlation, event-induced variance changes, and non-normality, so all things that make simpler tests unreliable.

The basic steps are:
1. **Standardized Abnormal Returns (SARs)** are computed for each event's estimation window by dividing the daily ARs by the estimation-window standard deviation.
2. A **Standardized CAR (SCAR)** is computed for the event window: `SCAR_i = CAR_i / (s_i * sqrt(L2))`
3. The estimation-period SARs and the event SCAR are combined into a single vector, ranked, and transformed to U-scores.
4. The test statistic is based on the cross-sectional average of the event-period U-score.

For small samples (N < 30) a Z-test is used; otherwise the t-distribution version is applied, which has better finite-sample properties.

In [5]:
# Two-Sample Comparison Tests
def comparison_tests(trad_df, llm_df, event_type_label):
    # average LLM CARs per event (across model x prompt_variant)
    llm_agg = llm_df.groupby(["ticker", "event_date"])[CAR_COLUMNS].mean().reset_index()

    merged = pd.merge(
        trad_df[["ticker", "event_date"] + CAR_COLUMNS],
        llm_agg[["ticker", "event_date"] + CAR_COLUMNS],
        on=["ticker", "event_date"],
        suffixes=("_trad", "_llm"),
    )

    results = []
    for car_col in CAR_COLUMNS:
        trad_col = f"{car_col}_trad"
        llm_col = f"{car_col}_llm"

        valid = merged.dropna(subset=[trad_col, llm_col])
        if len(valid) < 2:
            continue

        trad_vals = valid[trad_col].values
        llm_vals = valid[llm_col].values
        diff = trad_vals - llm_vals

        # paired t-test
        t_stat, t_pval = stats.ttest_rel(trad_vals, llm_vals)

        # Wilcoxon signed-rank test
        try:
            w_stat, w_pval = stats.wilcoxon(diff)
        except ValueError:
            # all differences are zero
            w_stat, w_pval = np.nan, np.nan

        results.append({
            "group": f"{event_type_label} (trad vs LLM)",
            "test_type": "paired_t_test",
            "car_window": car_col,
            "N": len(valid),
            "mean_diff": np.mean(diff),
            "test_statistic": t_stat,
            "p_value": t_pval,
            "significant_5pct": "Yes" if t_pval < 0.05 else "No",
            "significant_1pct": "Yes" if t_pval < 0.01 else "No",
        })
        results.append({
            "group": f"{event_type_label} (trad vs LLM)",
            "test_type": "wilcoxon_signed_rank",
            "car_window": car_col,
            "N": len(valid),
            "mean_diff": np.mean(diff),
            "test_statistic": w_stat,
            "p_value": w_pval,
            "significant_5pct": "Yes" if w_pval < 0.05 else "No",
            "significant_1pct": "Yes" if w_pval < 0.01 else "No",
        })

    return results

## Two-Sample Comparison Tests (Traditional vs LLM)

Here it is tested whether the traditional and LLM-computed CARs are significantly different from each other. Since both methods operate on the same events, **paired** tests are used:
- **Paired t-test**: assumes the differences are roughly normally distributed
- **Wilcoxon signed-rank test**: non-parametric alternative that doesn't need the normality assumption

The LLM CARs are averaged across all model/prompt variants first, so each event has one traditional CAR and one (averaged) LLM CAR.

In [6]:
def print_data_sample_stats(trad_df, llm_df, portfolio_df, market_df):
    print("DATA SAMPLE DESCRIPTIVE STATISTICS")

    print(f"\nTraditional CARs: {len(trad_df)} events")
    for et in sorted(trad_df["event_type"].unique()):
        n = (trad_df["event_type"] == et).sum()
        print(f"  {et}: {n}")

    if len(llm_df) > 0:
        n_events = llm_df.groupby(["ticker", "event_date"]).ngroups
        print(f"\nLLM CARs: {len(llm_df)} rows ({n_events} unique events)")
        print(f"  Models: {sorted(llm_df['model'].unique())}")
        print(f"  Prompt variants: {sorted(llm_df['prompt_variant'].unique())}")
        for et in sorted(llm_df["event_type"].dropna().unique()):
            n = llm_df[llm_df["event_type"] == et].groupby(["ticker", "event_date"]).ngroups
            print(f"  {et}: {n} unique events")

    # portfolio returns
    tickers = [c for c in portfolio_df.columns if c != "date"]
    print(f"\nPortfolio returns:")
    print(f"  Date range: {portfolio_df['date'].min().date()} to {portfolio_df['date'].max().date()}")
    print(f"  Tickers: {len(tickers)}")
    total_cells = portfolio_df[tickers].size
    missing = portfolio_df[tickers].isna().sum().sum()
    print(f"  Missing values: {missing}/{total_cells} ({100*missing/total_cells:.1f}%)")

    # market returns
    print(f"\nMarket returns:")
    print(f"  Date range: {market_df['date'].min().date()} to {market_df['date'].max().date()}")
    print(f"  Observations: {len(market_df)}")
    print(f"  Mean: {market_df['market_return'].mean():.6f}")
    print(f"  Std:  {market_df['market_return'].std():.6f}")


def print_descriptive_table(desc_rows):
    if not desc_rows:
        return
    df = pd.DataFrame(desc_rows)
    for group, gdf in df.groupby("group", sort=False):
        print(f"\n--- {group} ---")
        display = gdf[["car_window", "N", "mean", "median", "std", "min", "max"]].copy()
        display["mean"] = display["mean"].map("{:.6f}".format)
        display["median"] = display["median"].map("{:.6f}".format)
        display["std"] = display["std"].map("{:.6f}".format)
        display["min"] = display["min"].map("{:.6f}".format)
        display["max"] = display["max"].map("{:.6f}".format)
        print(display.to_string(index=False))


def print_grank_table(grank_rows):
    if not grank_rows:
        return
    df = pd.DataFrame(grank_rows)

    # unify statistic column
    if "z_statistic" not in df.columns:
        df["z_statistic"] = np.nan
    if "t_statistic" not in df.columns:
        df["t_statistic"] = np.nan
    df["test_statistic"] = df["z_statistic"].fillna(df["t_statistic"])

    for group, gdf in df.groupby("group", sort=False):
        for tt, tdf in gdf.groupby("test_type", sort=False):
            print(f"\n--- {group} | H0: {tt} CAR = 0 ---")
            display = tdf[["car_window", "test_method", "N_valid", "L1",
                           "CAR_center", "test_statistic", "p_value",
                           "significant_5pct", "significant_1pct"]].copy()
            display["CAR_center"] = display["CAR_center"].map("{:.6f}".format)
            display["test_statistic"] = display["test_statistic"].map("{:.4f}".format)
            display["p_value"] = display["p_value"].map("{:.6f}".format)
            print(display.to_string(index=False))


def print_comparison_table(comp_rows):
    if not comp_rows:
        return
    df = pd.DataFrame(comp_rows)

    for group, gdf in df.groupby("group", sort=False):
        for tt, tdf in gdf.groupby("test_type", sort=False):
            print(f"\n--- {group} | {tt} ---")
            display = tdf[["car_window", "N", "mean_diff", "test_statistic",
                           "p_value", "significant_5pct", "significant_1pct"]].copy()
            display["mean_diff"] = display["mean_diff"].map("{:.6f}".format)
            display["test_statistic"] = display["test_statistic"].map("{:.4f}".format)
            display["p_value"] = display["p_value"].map("{:.6f}".format)
            print(display.to_string(index=False))


def validate_event_types(trad_df, llm_df):
    # missing event_type in LLM data
    missing = llm_df[llm_df["event_type"].isna()]
    if len(missing) > 0:
        pairs = missing[["ticker", "event_date"]].drop_duplicates()
        print(f"  WARNING: {len(pairs)} LLM event(s) have no event_type:")
        for _, r in pairs.iterrows():
            print(f"    {r['ticker']} {r['event_date']}")

    # mismatches on shared events
    trad_map = trad_df[["ticker", "event_date", "event_type"]].drop_duplicates()
    llm_map = llm_df[["ticker", "event_date", "event_type"]].dropna(subset=["event_type"]).drop_duplicates()
    merged = pd.merge(trad_map, llm_map, on=["ticker", "event_date"], suffixes=("_trad", "_llm"))
    mismatched = merged[merged["event_type_trad"] != merged["event_type_llm"]]
    if len(mismatched) > 0:
        print(f"  WARNING: {len(mismatched)} event(s) have mismatched event_type:")
        for _, r in mismatched.iterrows():
            print(f"    {r['ticker']} {r['event_date']}: trad={r['event_type_trad']} vs llm={r['event_type_llm']}")

    print("  Traditional event_type distribution:")
    for et, n in trad_df["event_type"].value_counts().items():
        print(f"    {et}: {n}")
    print("  LLM event_type distribution:")
    if len(llm_df) > 0:
        for et, n in llm_df["event_type"].value_counts(dropna=False).items():
            print(f"    {et}: {n}")
    else:
        print("    (no LLM data)")

## Run Everything

This cell ties it all together: the data is loaded, descriptive stats are computed, G-Rank tests and comparison tests are run, and everything is saved to `cross_sectional_results.csv`. It takes a minute or two because the G-Rank test has to recompute SARs for every event.

In [7]:
output_path = "cross_sectional_results.csv"
duplicate_handling = "both"

print("Loading data...")
trad_df, llm_df, portfolio_df, market_df, group_defs, comp_pairs = load_data(duplicate_handling)

print("\nValidating event types...")
validate_event_types(trad_df, llm_df)

trading_cal = _build_trading_calendar(market_df)

all_results = []

print("DESCRIPTIVE STATISTICS")

desc_rows = []
for group_id, (event_type, method) in group_defs.items():
    if method == "traditional":
        subset = trad_df[trad_df["event_type"] == event_type]
        desc_rows.extend(descriptive_stats(subset, CAR_COLUMNS, f"{group_id}: {event_type} (traditional)"))
    else:
        subset = llm_df[llm_df["event_type"] == event_type]
        desc_rows.extend(descriptive_stats(subset, CAR_COLUMNS, f"{group_id}: {event_type} (LLM)"))

print_descriptive_table(desc_rows)
for r in desc_rows:
    r["section"] = "descriptive"
all_results.extend(desc_rows)

# G-Rank Significance Tests
print("\n" + "=" * 80)
print("G-RANK SIGNIFICANCE TESTS (H0: CAR = 0)")
print("=" * 80)

grank_rows = []
for group_id, (event_type, method) in group_defs.items():
    label = f"{group_id}: {event_type} ({method})"
    print(f"\nProcessing {label}...")

    if method == "traditional":
        subset = trad_df[trad_df["event_type"] == event_type].copy()
        alpha_col, beta_col = "alpha", "beta"
    else:
        subset = llm_df[llm_df["event_type"] == event_type].copy()
        alpha_col, beta_col = "llm_alpha", "llm_beta"

    if len(subset) == 0:
        print(f"  Skipping — no events")
        continue

    results = run_grank_tests(subset, portfolio_df, market_df, alpha_col, beta_col, label, trading_cal)
    grank_rows.extend(results)

print_grank_table(grank_rows)
for r in grank_rows:
    r["section"] = "grank_test"
all_results.extend(grank_rows)

# Two-Sample Comparison Tests
print("\n" + "=" * 80)
print("PAIRED COMPARISON TESTS (Traditional vs LLM)")
print("=" * 80)

comp_rows = []
for trad_id, llm_id, event_type in comp_pairs:
    trad_subset = trad_df[trad_df["event_type"] == event_type]
    llm_subset = llm_df[llm_df["event_type"] == event_type]

    if len(trad_subset) == 0 or len(llm_subset) == 0:
        print(f"\n  Skipping {event_type} — no data in one or both groups")
        continue

    results = comparison_tests(trad_subset, llm_subset, event_type)
    comp_rows.extend(results)

print_comparison_table(comp_rows)
for r in comp_rows:
    r["section"] = "comparison_test"
all_results.extend(comp_rows)

# Data Sample Statistics
print_data_sample_stats(trad_df, llm_df, portfolio_df, market_df)

if all_results:
    out_df = pd.DataFrame(all_results)
    out_df.to_csv(output_path, index=False)
    print(f"\nSaved {len(out_df)} result rows to {output_path}")
else:
    print("\nNo results to save.")

Loading data...
Loaded 984 rows from cumulative_abnormal_returns_both.csv
  Traditional: 492 rows, LLM: 492 rows

Validating event types...
    ADI 2025-03-06 00:00:00: trad=S - Sale vs llm=S - Sale+OE
    ADI 2025-03-06 00:00:00: trad=S - Sale+OE vs llm=S - Sale
    ADP 2025-01-06 00:00:00: trad=S - Sale+OE vs llm=S - Sale
    ADP 2025-01-06 00:00:00: trad=S - Sale vs llm=S - Sale+OE
    ADP 2025-02-04 00:00:00: trad=S - Sale vs llm=S - Sale+OE
    ADP 2025-02-04 00:00:00: trad=S - Sale+OE vs llm=S - Sale
    AMZN 2024-11-25 00:00:00: trad=S - Sale vs llm=S - Sale+OE
    AMZN 2024-11-25 00:00:00: trad=S - Sale+OE vs llm=S - Sale
    AMZN 2025-02-25 00:00:00: trad=S - Sale vs llm=S - Sale+OE
    AMZN 2025-02-25 00:00:00: trad=S - Sale+OE vs llm=S - Sale
    ANET 2024-11-05 00:00:00: trad=S - Sale+OE vs llm=S - Sale
    ANET 2024-11-05 00:00:00: trad=S - Sale vs llm=S - Sale+OE
    ANET 2025-04-09 00:00:00: trad=S - Sale+OE vs llm=S - Sale
    ANET 2025-04-09 00:00:00: trad=S - Sale vs 

Mean CARs across all post-event windows are close to zero for both Sale groups, with the pre-event window CAR[-20,-1] showing small positive drift (+1.5% for Sale, +2.5% for Sale+OE). Purchases show weakly negative CARs across most windows, but with only N=12 events and large standard deviations these numbers are essentially noise. The LLM-computed CARs follow a similar pattern but tend to have slightly larger magnitudes and wider standard deviations than their traditional counterparts.

The G-Rank significance tests paint a clear picture. For Purchase events, none of the individual CAR windows reach significance at the 5% level except for the full window CAR[-20,+20] (p=0.033), which is likely an artefact of the small sample (N=12). For Sale events, only the pre-event drift CAR[-20,-1] is significant (p=0.038), suggesting some information leakage or anticipatory trading before the filing date. Sale+OE events show a similar but stronger pattern: the pre-event drift is significant at the 1% level (p=0.005) and the full window also reaches significance (p=0.011). Crucially, none of the post-event windows (CAR[0,1] through CAR[0,20]) are individually significant for any trade type, meaning there is no robust evidence of abnormal returns following insider trade filings.

The paired comparison tests between traditional and LLM CARs show no significant differences for Purchase events. For Sale and Sale+OE events, the short-term windows CAR[0,1] and CAR[0,5] are not significantly different either, suggesting that the LLM approach tracks the traditional method well in the immediate post-event period. However, the pre-event window and longer post-event windows (CAR[0,20], CAR[-20,+20]) show statistically significant divergence (p < 0.01 in several cases). This is consistent with small estimation-error differences in alpha and beta compounding over more trading days.